# PDHG Original Run With a Final \(1/\sqrt{j}\) Tail (Colab)

This notebook launches one phase-retrieval PDHG run. Its default preset reproduces the supplied run through iteration \(461\), including the original 53-step tail, and then replaces only iterations \(462,\ldots,471\) with a second 10-step tail.

The original tail starts at \(s=419\), with \(t=k-s\):

\[
a_t=\left(\frac{c}{c+t}\right)^{1/2},\qquad
\sigma_k=\sigma_s a_t,\qquad
\tau_k=\frac{\sigma_k^2}{\lambda},\qquad
\rho_k=\sigma_s a_t^{1+q}.
\]

The final stage is re-anchored at \(k=462\). With \(j=k-461\in\{1,\ldots,10\}\), its default is

\[
\sigma_k=\frac{\sigma_{\rm anchor}}{\sqrt{j}},\qquad
\tau_k=\frac{\sigma_k^2}{\lambda},\qquad
\rho_k=\frac{\sigma_{\rm anchor}}{j}.
\]

This is implemented as a separate final tail with \(c_{\rm final}=1\) and \(q_{\rm final}=1\). The phase-retrieval dual-step continuation \(\gamma_k\) remains separate and is recomputed from the resulting \(\sigma_k\) sequence.

## Exact default comparison

The default controls reproduce the supplied configuration:

- Configured maximum iterations: \(500\), with effective stopping at \(K=472\).
- Prefix scheduler: 500 entries, \(\sigma_{\max}=10\), \(\sigma_{\min}=0.075\), and poly-7 timesteps.
- Original tail: 53 entries beginning at \(k=419\), \(c=33.8025084713\), raw rho power \(1.1\), and explicit \(\lambda=2.0103451448\).
- Final experiment: only the last 10 entries, beginning at \(k=462\), use \(1/\sqrt{j}\).
- Phase dual schedule: \(\gamma_0=1100\), inverse-square scaling over the last 100 effective iterations.

The final tail reuses the original \(\lambda\) for \(\tau_k=\sigma_k^2/\lambda\). Its default rho scale is 1, which gives the literal \(\rho_k=\sigma_{\rm anchor}/j\). Set FINAL_TAIL_RHO_SCALE to \(0.959795426757\) if you instead want rho to be continuous at \(k=462\).

The implementation fields use rho power \(1+q\). Therefore the original \(q=0.1\) becomes raw power \(1.1\), while the final \(q=1\) becomes raw power \(2\).

In [ ]:
#@title Project and run settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_final_tail_single_run_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
RUN_NAME = "Inverse_PR_Original_Plus_Final_RootK"  #@param {type:"string"}
CONFIG_NAME = "default_ffhq.yaml"  #@param {type:"string"}

INVERSE_TASK = "phase_retrieval"  #@param ["phase_retrieval", "inpainting", "inpainting_rand", "motion_blur", "gaussian_blur", "down_sampling", "down_sampling_explicit", "hdr", "nonlinear_blur", "compression_quantization"]
SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}

# Exact prefix and effective run length from the supplied run.
MAX_ITER = 500  #@param {type:"integer"}
EARLY_STOP = 472  #@param {type:"integer"}
SIGMA_MAX = 10.0  #@param {type:"number"}
PREFIX_NUM_STEPS = 500  #@param {type:"integer"}
PREFIX_TIMESTEP = "poly-7"  #@param ["poly-1", "poly-7"]
PREFIX_SIGMA_MIN = 0.075  #@param {type:"number"}

# Original 53-step tail from the supplied run.
TAIL_STEPS = 53  #@param {type:"integer"}
TAIL_C = 33.8025084713  #@param {type:"number"}
TAIL_Q = 0.1  #@param {type:"number"}
TAIL_RHO_SCALE = 1.0  #@param {type:"number"}
TAIL_PARAMETERIZATION = "prefix"  #@param ["prefix", "sigma_s", "lambda"]
TAU0 = 0.01  #@param {type:"number"}
SIGMA_S = 0.214791394635  #@param {type:"number"}
USE_EXPLICIT_TAIL_LAMBDA = True  #@param {type:"boolean"}
LAMBDA_EFF = 2.0103451448  #@param {type:"number"}

# Independent final-stage override. c=1 means sigma=sigma_anchor/sqrt(j).
FINAL_TAIL_ACTIVATE = True  #@param {type:"boolean"}
FINAL_TAIL_STEPS = 10  #@param {type:"integer"}
FINAL_TAIL_C = 1.0  #@param {type:"number"}
FINAL_TAIL_Q = 1.0  #@param {type:"number"}
FINAL_TAIL_RHO_SCALE = 1.0  #@param {type:"number"}
FINAL_TAIL_LAMBDA_EFF = 2.0103451448  #@param {type:"number"}

# Phase-retrieval gamma_k (the implementation calls this sigma_dual).
GAMMA0 = 1100.0  #@param {type:"number"}
GAMMA_SCHEDULE_MODE = "to_infinity"  #@param ["constant", "to_zero", "to_infinity"]
GAMMA_SCHEDULE_SCOPE = "tail"  #@param ["tail", "full"]
GAMMA_SCHEDULE_TAIL_STEPS = 100  #@param {type:"integer"}
GAMMA_SCHEDULE_POWER = 2.0  #@param {type:"number"}
GAMMA_SCHEDULE_MIN = 1e-08  #@param {type:"number"}
GAMMA_SCHEDULE_MAX = 0.0  #@param {type:"number"}

DENOISER_AC_NOISE = True  #@param {type:"boolean"}
DENOISE_FINAL_STEP = "tweedie"  #@param ["tweedie", "ode"]
PHASE_DUAL_ACTIVE_Y_EPS = 0.001  #@param {type:"number"}
PHASE_PR_ALPHA = 0.25  #@param {type:"number"}
EVAL_METRICS = "psnr;ssim;lpips"  #@param {type:"string"}
SAVE_SAMPLES = False  #@param {type:"boolean"}
SAVE_TRAJ = False  #@param {type:"boolean"}
SAVE_TRAJ_RAW_DATA = False  #@param {type:"boolean"}
PROGRESS_JSON_EVERY = 1  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

# Optional Hydra overrides separated by semicolons.
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}

In [ ]:
#@title Preview and validate the exact two-stage schedule
import json
import math

def build_schedule_plan():
    max_iter = int(MAX_ITER)
    early_stop = int(EARLY_STOP)
    K = min(max_iter, early_stop) if early_stop > 0 else max_iter
    scheduler_num_steps = int(PREFIX_NUM_STEPS)
    tail_steps = int(TAIL_STEPS)
    tau0 = float(TAU0)
    sigma_max = float(SIGMA_MAX)
    prefix_sigma_min = float(PREFIX_SIGMA_MIN)
    c = float(TAIL_C)
    q = float(TAIL_Q)
    rho_scale = float(TAIL_RHO_SCALE)

    if K < 3:
        raise ValueError("The effective run length must be at least 3.")
    if max_iter < K:
        raise ValueError("MAX_ITER cannot be smaller than the effective run length.")
    if scheduler_num_steps < 2:
        raise ValueError("PREFIX_NUM_STEPS must be at least 2.")
    if tail_steps < 2 or tail_steps >= K:
        raise ValueError("TAIL_STEPS must lie in [2, K - 1].")
    if tau0 <= 0 or sigma_max <= 0 or c <= 0 or rho_scale <= 0:
        raise ValueError("TAU0, SIGMA_MAX, TAIL_C, and TAIL_RHO_SCALE must be positive.")
    if prefix_sigma_min <= 0 or prefix_sigma_min >= sigma_max:
        raise ValueError("PREFIX_SIGMA_MIN must lie strictly between 0 and SIGMA_MAX.")
    if not PREFIX_TIMESTEP.startswith("poly-"):
        raise ValueError("This notebook supports poly-n prefix timesteps only.")
    prefix_power = int(PREFIX_TIMESTEP.split("-", 1)[1])
    if prefix_power <= 0:
        raise ValueError("The poly-n prefix power must be positive.")

    switch_index = K - tail_steps
    mode = str(TAIL_PARAMETERIZATION).strip().lower()
    sigma_target = None
    if mode == "sigma_s":
        sigma_target = float(SIGMA_S)
        if sigma_target <= 0:
            raise ValueError("SIGMA_S must be positive.")
        scheduler_num_steps = switch_index + 1
        scheduler_sigma_min = sigma_target
    elif mode == "lambda":
        requested_lambda = float(LAMBDA_EFF)
        if requested_lambda <= 0:
            raise ValueError("LAMBDA_EFF must be positive.")
        sigma_target = math.sqrt(requested_lambda * tau0)
        scheduler_num_steps = switch_index + 1
        scheduler_sigma_min = sigma_target
    elif mode == "prefix":
        scheduler_sigma_min = prefix_sigma_min
    else:
        raise ValueError("TAIL_PARAMETERIZATION must be prefix, sigma_s, or lambda.")

    def sigma_at(index):
        clamped_index = min(index, scheduler_num_steps - 1)
        r = clamped_index / float(scheduler_num_steps - 1)
        hi = sigma_max ** (1.0 / prefix_power)
        lo = scheduler_sigma_min ** (1.0 / prefix_power)
        return ((1.0 - r) * hi + r * lo) ** prefix_power

    base_sigma = [sigma_at(k) for k in range(K)]
    sigma_switch = base_sigma[switch_index]
    if sigma_target is not None and not math.isclose(
        sigma_switch, sigma_target, rel_tol=1e-10, abs_tol=1e-12
    ):
        raise RuntimeError("Internal sigma_s targeting check failed.")

    implicit_lambda = sigma_switch ** 2 / tau0
    use_explicit_lambda = bool(USE_EXPLICIT_TAIL_LAMBDA) or mode == "lambda"
    explicit_lambda = float(LAMBDA_EFF) if use_explicit_lambda else None
    if explicit_lambda is not None and explicit_lambda <= 0:
        raise ValueError("LAMBDA_EFF must be positive when explicit coupling is enabled.")

    sigma = list(base_sigma)
    tau = [tau0] * K
    rho = list(base_sigma)
    tail_mask = [False] * K
    final_tail_mask = [False] * K
    rho_power_for_code = 1.0 + q

    for k in range(switch_index, K):
        t = k - switch_index
        a = math.sqrt(c / (c + t))
        sigma_k = sigma_switch * a
        tau_k = sigma_k ** 2 / explicit_lambda if explicit_lambda is not None else tau0 * a ** 2
        rho_k = rho_scale * sigma_switch * a ** rho_power_for_code
        sigma[k] = sigma_k
        tau[k] = tau_k
        rho[k] = rho_k
        tail_mask[k] = True

    final_tail_steps = int(FINAL_TAIL_STEPS) if bool(FINAL_TAIL_ACTIVATE) else 0
    if final_tail_steps < 0 or final_tail_steps > tail_steps:
        raise ValueError("FINAL_TAIL_STEPS must lie in [0, TAIL_STEPS].")

    final_switch_index = None
    final_sigma_anchor = None
    final_tau_before = None
    final_rho_before = None
    final_rho_continuity_scale = None
    final_rho_power_for_code = 1.0 + float(FINAL_TAIL_Q)
    if final_tail_steps > 0:
        final_switch_index = K - final_tail_steps
        final_sigma_anchor = sigma[final_switch_index]
        final_tau_before = tau[final_switch_index]
        final_rho_before = rho[final_switch_index]
        final_rho_continuity_scale = final_rho_before / final_sigma_anchor
        final_c = float(FINAL_TAIL_C)
        final_lambda = float(FINAL_TAIL_LAMBDA_EFF)
        final_rho_scale = float(FINAL_TAIL_RHO_SCALE)
        if final_c <= 0 or final_lambda <= 0 or final_rho_scale <= 0:
            raise ValueError("Final-tail c, lambda, and rho scale must be positive.")

        for k in range(final_switch_index, K):
            n = k - final_switch_index
            ratio = math.sqrt(final_c / (final_c + n))
            sigma_k = final_sigma_anchor * ratio
            sigma[k] = sigma_k
            tau[k] = sigma_k ** 2 / final_lambda
            rho[k] = final_rho_scale * final_sigma_anchor * ratio ** final_rho_power_for_code
            final_tail_mask[k] = True

    tail_rows = []
    for k in range(switch_index, K):
        tail_rows.append({
            "k": k,
            "stage": "final" if final_tail_mask[k] else "original",
            "sigma": sigma[k],
            "tau": tau[k],
            "rho": rho[k],
        })
    final_tail_rows = [row for row in tail_rows if row["stage"] == "final"]

    gamma = [float(GAMMA0)] * K
    gamma_mode = str(GAMMA_SCHEDULE_MODE)
    gamma_tail_steps = int(GAMMA_SCHEDULE_TAIL_STEPS)
    if gamma_tail_steps < 0 or gamma_tail_steps > K:
        raise ValueError("GAMMA_SCHEDULE_TAIL_STEPS must lie in [0, K].")
    gamma_anchor_index = None
    if gamma_mode != "constant":
        if gamma_mode not in {"to_zero", "to_infinity"}:
            raise ValueError("Unsupported GAMMA_SCHEDULE_MODE.")
        if GAMMA_SCHEDULE_SCOPE == "tail":
            gamma_anchor_index = K - gamma_tail_steps if gamma_tail_steps > 0 else switch_index
        else:
            gamma_anchor_index = 0
        anchor_sigma = sigma[gamma_anchor_index]
        gamma_power = float(GAMMA_SCHEDULE_POWER)
        gamma_min = float(GAMMA_SCHEDULE_MIN) if float(GAMMA_SCHEDULE_MIN) > 0 else None
        gamma_max = float(GAMMA_SCHEDULE_MAX) if float(GAMMA_SCHEDULE_MAX) > 0 else None
        for k in range(gamma_anchor_index, K):
            ratio = sigma[k] / anchor_sigma
            value = float(GAMMA0) * (
                ratio ** gamma_power if gamma_mode == "to_zero" else ratio ** (-gamma_power)
            )
            if gamma_min is not None:
                value = max(value, gamma_min)
            if gamma_max is not None:
                value = min(value, gamma_max)
            gamma[k] = value

    return {
        "max_iter": max_iter,
        "early_stop": early_stop,
        "K": K,
        "tail_steps": tail_steps,
        "switch_index": switch_index,
        "prefix_timestep": PREFIX_TIMESTEP,
        "scheduler_num_steps": scheduler_num_steps,
        "scheduler_sigma_min": scheduler_sigma_min,
        "sigma_s": sigma_switch,
        "tau0": tau0,
        "implicit_lambda": implicit_lambda,
        "explicit_tail_lambda": explicit_lambda,
        "c": c,
        "q": q,
        "rho_scale": rho_scale,
        "rho_power_for_code": rho_power_for_code,
        "final_tail_steps": final_tail_steps,
        "final_switch_index": final_switch_index,
        "final_sigma_anchor": final_sigma_anchor,
        "final_tau_before": final_tau_before,
        "final_rho_before": final_rho_before,
        "final_rho_continuity_scale": final_rho_continuity_scale,
        "final_c": float(FINAL_TAIL_C),
        "final_lambda": float(FINAL_TAIL_LAMBDA_EFF),
        "final_q": float(FINAL_TAIL_Q),
        "final_rho_scale": float(FINAL_TAIL_RHO_SCALE),
        "final_rho_power_for_code": final_rho_power_for_code,
        "sigma": sigma,
        "tau": tau,
        "rho": rho,
        "gamma": gamma,
        "gamma_tail_steps": gamma_tail_steps,
        "gamma_anchor_index": gamma_anchor_index,
        "tail_mask": tail_mask,
        "final_tail_mask": final_tail_mask,
        "tail_rows": tail_rows,
        "final_tail_rows": final_tail_rows,
    }

schedule_plan = build_schedule_plan()

summary_keys = [
    "max_iter", "early_stop", "K", "scheduler_num_steps", "scheduler_sigma_min",
    "tail_steps", "switch_index", "sigma_s", "tau0", "implicit_lambda",
    "explicit_tail_lambda", "c", "q", "rho_power_for_code",
    "final_tail_steps", "final_switch_index", "final_sigma_anchor",
    "final_tau_before", "final_rho_before", "final_rho_continuity_scale",
    "final_c", "final_lambda", "final_q", "final_rho_scale",
    "final_rho_power_for_code", "gamma_tail_steps", "gamma_anchor_index",
]
print(json.dumps({key: schedule_plan[key] for key in summary_keys}, indent=2))
print()
print("Rows around the final-tail switch:")
final_switch = schedule_plan["final_switch_index"]
for row in schedule_plan["tail_rows"]:
    if final_switch is not None and final_switch - 2 <= row["k"] <= final_switch + 2:
        print(row)
print()
print("Last three rows:")
for row in schedule_plan["tail_rows"][-3:]:
    print(row)

try:
    import matplotlib.pyplot as plt
    k_values = list(range(schedule_plan["K"]))
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    for ax, key, label in [
        (axes[0, 0], "sigma", "sigma_k"),
        (axes[0, 1], "tau", "tau_k"),
        (axes[1, 0], "rho", "rho_k"),
        (axes[1, 1], "gamma", "gamma_k"),
    ]:
        ax.plot(k_values, schedule_plan[key])
        ax.axvline(schedule_plan["switch_index"], color="tab:red", linestyle="--", alpha=0.7)
        if schedule_plan["final_switch_index"] is not None:
            ax.axvline(
                schedule_plan["final_switch_index"],
                color="tab:purple",
                linestyle=":",
                alpha=0.9,
            )
        ax.set_title(label)
        ax.set_xlabel("iteration k")
        ax.grid(alpha=0.25)
    plt.show()
except ImportError:
    print("matplotlib is unavailable; skipping the plot.")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Fetch the repository
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo_dir.as_posix()],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repository ready: {repo_dir}")

In [ ]:
#@title Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")

In [ ]:
#@title Download the FFHQ checkpoint if needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
checkpoint_path = Path("pretrained-models/ffhq_10m.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

if checkpoint_path.exists():
    print(f"Checkpoint already present: {checkpoint_path}")
else:
    subprocess.run(
        ["gdown", "--id", "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh", "-O", checkpoint_path.as_posix()],
        check=True,
    )
    print(f"Downloaded checkpoint to: {checkpoint_path}")

In [ ]:
#@title Build the single-run command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

def parse_semicolon_list(text):
    return [
        item.strip()
        for item in str(text).replace("\n", ";").split(";")
        if item.strip()
    ]

metric_list = parse_semicolon_list(EVAL_METRICS)
if not metric_list:
    raise ValueError("EVAL_METRICS must contain at least one metric.")
extra_overrides = parse_semicolon_list(EXTRA_HYDRA_OVERRIDES)

schedule_plan = build_schedule_plan()
session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
run_tag = f"pdhg_final_rootk_{INVERSE_TASK}_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"
save_root = repo_dir / "results" / "single_runs" / run_tag
hydra_root = repo_dir / "outputs" / "single_runs" / run_tag
run_aux_root = repo_dir / "single_runs"
latest_log_path = run_aux_root / f"{run_tag}.log"
latest_pid_path = run_aux_root / f"{run_tag}.pid"
progress_path = run_aux_root / f"{run_tag}.progress.json"
context_path = run_aux_root / f"{run_tag}.context.json"
schedule_manifest_path = run_aux_root / f"{run_tag}.schedule.json"
data_end_idx = int(DATA_START_IDX) + int(TOTAL_IMAGES)

run_cmd = [
    PYTHON_BIN,
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    "sampler=edm_pdhg",
    f"inverse_task={INVERSE_TASK}",
    f"name={run_name}",
    f"seed={int(SEED)}",
    "gpu=0",
    "wandb=false",
    "show_config=false",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    f"save_traj={'true' if SAVE_TRAJ else 'false'}",
    f"save_traj_raw_data={'true' if SAVE_TRAJ_RAW_DATA else 'false'}",
    f"total_images={int(TOTAL_IMAGES)}",
    f"batch_size={int(BATCH_SIZE)}",
    "num_runs=1",
    f"inverse_task.operator.sigma={float(MEASUREMENT_SIGMA)}",
    f"sampler.annealing_scheduler_config.num_steps={schedule_plan['scheduler_num_steps']}",
    f"sampler.annealing_scheduler_config.sigma_max={float(SIGMA_MAX)}",
    f"sampler.annealing_scheduler_config.sigma_min={schedule_plan['scheduler_sigma_min']}",
    "sampler.annealing_scheduler_config.schedule=linear",
    f"sampler.annealing_scheduler_config.timestep={PREFIX_TIMESTEP}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_steps={schedule_plan['tail_steps']}",
    "++sampler.annealing_scheduler_config.theorem1_tail_sigma_mode=theorem1",
    f"++sampler.annealing_scheduler_config.theorem1_tail_c={schedule_plan['c']}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_rho_power={schedule_plan['rho_power_for_code']}",
    f"++sampler.annealing_scheduler_config.theorem1_tail_rho_scale={schedule_plan['rho_scale']}",
    f"inverse_task.admm_config.max_iter={schedule_plan['max_iter']}",
    f"++inverse_task.admm_config.early_stop={schedule_plan['early_stop']}",
    f"++inverse_task.admm_config.pdhg.tau={schedule_plan['tau0']}",
    f"++inverse_task.admm_config.pdhg.sigma_dual={float(GAMMA0)}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_mode={GAMMA_SCHEDULE_MODE}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_scope={GAMMA_SCHEDULE_SCOPE}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_tail_steps={int(GAMMA_SCHEDULE_TAIL_STEPS)}",
    f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_power={float(GAMMA_SCHEDULE_POWER)}",
    f"inverse_task.admm_config.denoise.final_step={DENOISE_FINAL_STEP}",
    f"++inverse_task.admm_config.denoise.ac_noise={'true' if DENOISER_AC_NOISE else 'false'}",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
    f"++inverse_task.admm_config.pdhg.phase_dual_active_y_eps={float(PHASE_DUAL_ACTIVE_Y_EPS)}",
    f"++inverse_task.admm_config.pdhg.phase_pr_alpha={float(PHASE_PR_ALPHA)}",
    f"eval_fn_list=[{','.join(metric_list)}]",
    f"data.image_root_path={drive_data_dir.as_posix()}",
    f"data.start_idx={int(DATA_START_IDX)}",
    f"data.end_idx={data_end_idx}",
    f"++progress_json_path={progress_path.as_posix()}",
    f"++progress_json_every={max(1, int(PROGRESS_JSON_EVERY))}",
]

if schedule_plan["explicit_tail_lambda"] is not None:
    run_cmd.append(
        f"++sampler.annealing_scheduler_config.theorem1_tail_lambda={schedule_plan['explicit_tail_lambda']}"
    )
if schedule_plan["final_tail_steps"] > 0:
    run_cmd.extend([
        f"++sampler.annealing_scheduler_config.final_tail_steps={schedule_plan['final_tail_steps']}",
        f"++sampler.annealing_scheduler_config.final_tail_c={schedule_plan['final_c']}",
        f"++sampler.annealing_scheduler_config.final_tail_lambda={schedule_plan['final_lambda']}",
        f"++sampler.annealing_scheduler_config.final_tail_rho_power={schedule_plan['final_rho_power_for_code']}",
        f"++sampler.annealing_scheduler_config.final_tail_rho_scale={schedule_plan['final_rho_scale']}",
    ])
if float(GAMMA_SCHEDULE_MIN) > 0:
    run_cmd.append(
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_min={float(GAMMA_SCHEDULE_MIN)}"
    )
if float(GAMMA_SCHEDULE_MAX) > 0:
    run_cmd.append(
        f"++inverse_task.admm_config.pdhg.sigma_dual_schedule_max={float(GAMMA_SCHEDULE_MAX)}"
    )

run_cmd.extend(extra_overrides)
run_cmd.extend([
    f"save_dir={save_root.as_posix()}",
    f"hydra.run.dir={hydra_root.as_posix()}",
])

run_aux_root.mkdir(parents=True, exist_ok=True)
manifest = {
    "formulas": {
        "original_a_t": "sqrt(c / (c + t))",
        "original_sigma": "sigma_s * a_t",
        "original_tau": "sigma^2 / lambda when explicit lambda is enabled",
        "original_rho": "rho_scale * sigma_s * a_t^(1 + q)",
        "final_ratio": "sqrt(final_c / (final_c + n)); final_c=1 gives 1/sqrt(j)",
        "final_sigma": "final_sigma_anchor * final_ratio",
        "final_tau": "final_sigma^2 / final_lambda",
        "final_rho": "final_rho_scale * final_sigma_anchor * final_ratio^(1 + final_q)",
    },
    **schedule_plan,
}
schedule_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

last_context = {
    "run_tag": run_tag,
    "run_name": run_name,
    "save_root": save_root.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_progress_path": progress_path.as_posix(),
    "context_path": context_path.as_posix(),
    "schedule_manifest_path": schedule_manifest_path.as_posix(),
    "run_cmd": run_cmd,
}
context_path.write_text(json.dumps(last_context, indent=2), encoding="utf-8")

print(f"Run tag: {run_tag}")
print(f"Task: {INVERSE_TASK}")
print(f"Dataset slice: [{DATA_START_IDX}, {data_end_idx})")
print(f"Configured max_iter: {schedule_plan['max_iter']}")
print(f"Effective K: {schedule_plan['K']}")
print(f"Prefix scheduler entries: {schedule_plan['scheduler_num_steps']}")
print(f"Original tail switch: {schedule_plan['switch_index']}")
print(f"Original tail entries: {schedule_plan['tail_steps']}")
print(f"Original sigma_s: {schedule_plan['sigma_s']:.12g}")
print(f"Original explicit lambda: {schedule_plan['explicit_tail_lambda']}")
print(f"Final tail switch: {schedule_plan['final_switch_index']}")
print(f"Final tail entries: {schedule_plan['final_tail_steps']}")
print(f"Final sigma anchor: {schedule_plan['final_sigma_anchor']:.12g}")
print(f"Final rho continuity scale: {schedule_plan['final_rho_continuity_scale']:.12g}")
print(f"Final sigma: {schedule_plan['sigma'][-1]:.12g}")
print(f"Final tau: {schedule_plan['tau'][-1]:.12g}")
print(f"Final rho: {schedule_plan['rho'][-1]:.12g}")
print(f"Final gamma: {schedule_plan['gamma'][-1]:.12g}")
print(f"Save root: {save_root}")
print(f"Schedule manifest: {schedule_manifest_path}")
if extra_overrides:
    print(f"Extra overrides: {extra_overrides}")
print("\nCommand:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

In [ ]:
#@title Validate the Hydra configuration without running the model
import subprocess

validation_cmd = last_context["run_cmd"][:4] + ["--cfg", "job"] + last_context["run_cmd"][4:]
validation = subprocess.run(
    validation_cmd,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(validation.stdout[-12000:])
if validation.returncode != 0:
    raise RuntimeError(f"Hydra validation failed with exit code {validation.returncode}")
print("Hydra configuration is valid.")

In [ ]:
#@title Launch the run in the background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
log_path = Path(last_context["latest_log_path"])
pid_path = Path(last_context["latest_pid_path"])
log_path.parent.mkdir(parents=True, exist_ok=True)

with log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {log_path}")
print(f"Progress: {last_context['run_progress_path']}")
print(f"Save root: {last_context['save_root']}")

In [ ]:
#@title Show recent log lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
print("\n".join(lines[-int(LOG_TAIL_LINES):]) if lines else "<log is empty>")

In [ ]:
#@title Show live progress and current schedule values
import json
from pathlib import Path

progress_path = Path(last_context["run_progress_path"])
if not progress_path.exists():
    raise FileNotFoundError(f"Progress file not found yet: {progress_path}")

payload = json.loads(progress_path.read_text(encoding="utf-8"))
print(json.dumps(payload, indent=2))

step = payload.get("step")
if isinstance(step, int) and 1 <= step <= schedule_plan["K"]:
    index = step - 1
    print()
    print("Schedule at the latest completed iteration:")
    print({
        "k": index,
        "original_tail_active": schedule_plan["tail_mask"][index],
        "final_tail_active": schedule_plan["final_tail_mask"][index],
        "sigma": schedule_plan["sigma"][index],
        "tau": schedule_plan["tau"][index],
        "rho": schedule_plan["rho"][index],
        "gamma": schedule_plan["gamma"][index],
    })

In [ ]:
#@title Show results and recorded schedule tails
import json
from pathlib import Path

save_root = Path(last_context["save_root"])
metrics_matches = sorted(save_root.rglob("metrics.json"))
history_matches = sorted(save_root.rglob("metric_history.json"))
eval_matches = sorted(save_root.rglob("eval.md"))
grid_matches = sorted(save_root.rglob("grid_results.png"))

print(f"save_root: {save_root}")
print(f"metrics files: {[p.as_posix() for p in metrics_matches]}")
print(f"history files: {[p.as_posix() for p in history_matches]}")
print(f"evaluation files: {[p.as_posix() for p in eval_matches]}")
print(f"image grids: {[p.as_posix() for p in grid_matches]}")

if metrics_matches:
    print()
    print("Final metrics:")
    print(json.dumps(json.loads(metrics_matches[0].read_text(encoding="utf-8")), indent=2))

if history_matches:
    history = json.loads(history_matches[0].read_text(encoding="utf-8"))
    history_view = history.get("runs", [history])[0] if isinstance(history, dict) else {}
    print()
    print("Recorded final schedule values:")
    for key in ["sigma", "tau", "rho", "sigma_dual"]:
        values = history_view.get(key, []) if isinstance(history_view, dict) else []
        if values:
            print(f"{key}: len={len(values)} tail={values[-min(10, len(values)):]}")

if eval_matches:
    print()
    print(eval_matches[0].read_text(encoding="utf-8", errors="ignore"))

In [ ]:
#@title Copy run artifacts to Google Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["save_root"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
    Path(last_context["context_path"]),
    Path(last_context["schedule_manifest_path"]),
    Path(last_context["run_progress_path"]),
]

for source in targets:
    if not source.exists():
        print(f"Skipping missing path: {source}")
        continue
    destination = export_root / source.name
    if source.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print(f"Copied {source} -> {destination}")